# 1 — Quickstart

**Runs on CPU in well under a minute. No model weights, no dataset download.**

The point of this notebook is not to impress you with a result — it is to get a FOCAL panel into
your hands and show you the three objects the API returns, so the next four tutorials have
something to build on.

We use synthetic data and `StubEncoder`, a deterministic stand-in encoder
(`embed = L2_normalize(log1p(X) @ W)`, `W = I`). It is not a foundation model and makes no
biological claims. What it *is* good for is showing the shape of the computation with nothing
hidden: the contrast, the ranking, and the gate are all visible in a toy you can reason about by
hand.

:::{note}
`StubEncoder` lives in `focal.encoders`, which imports torch unconditionally — so even this
CPU-only notebook needs the `[attribution]` install tier, not the core one.
:::

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")  # no ipywidgets in this kernel

import numpy as np, pandas as pd, anndata as ad
import focal
from focal.encoders import StubEncoder

print("focal", focal.__version__)

focal 0.5.0


## A toy where we know the answer

Three populations — `Alpha`, `Beta`, `Gamma` — of 80 cells each, over 39 genes with a deliberate
structure:

- **5 housekeeping genes** (`HK1`–`HK5`), expressed *very* highly in all three populations.
- **3 marker genes per population** (`AMARK1`–`3`, `BMARK…`, `GMARK…`), moderately expressed in
  their own population and near-absent elsewhere.
- **25 noise genes**, low and uninformative everywhere.

The markers are the right answer. The housekeeping genes are the trap: they are by far the
highest-expressed genes in every population, and they say nothing about which population a cell
belongs to.

In [2]:
rng = np.random.default_rng(0)
states = {"Alpha": 80, "Beta": 80, "Gamma": 80}
hk     = [f"HK{i}" for i in range(1, 6)]
marks  = {s: [f"{s[0].upper()}MARK{i}" for i in (1, 2, 3)] for s in states}
noise  = [f"NOISE{i}" for i in range(1, 26)]
genes  = hk + [g for s in states for g in marks[s]] + noise

blocks, labels = [], []
for s, n in states.items():
    B = np.zeros((n, len(genes)))
    for j, g in enumerate(genes):
        if   g in hk:        B[:, j] = rng.poisson(60, n)   # high everywhere -- the trap
        elif g in marks[s]:  B[:, j] = rng.poisson(12, n)   # this population's markers
        elif any(g in v for v in marks.values()):
            B[:, j] = rng.poisson(1, n)                     # another population's markers
        else:                B[:, j] = rng.poisson(4, n)    # noise
    blocks.append(B); labels += [s] * n

adata = ad.AnnData(np.vstack(blocks).astype("float32"))
adata.var_names = genes
adata.obs["state"] = pd.Categorical(labels)
adata

AnnData object with n_obs × n_vars = 240 × 39
    obs: 'state'

## First, what plain expression says

Rank genes for `Alpha` by mean expression — the thing a naive "top expressed genes" readout gives
you.

In [3]:
means = pd.DataFrame(adata.X, columns=genes, index=labels).groupby(level=0).mean()
means.loc["Alpha"].sort_values(ascending=False).head(8).round(2)

HK2       61.490002
HK1       61.259998
HK4       61.209999
HK3       60.340000
HK5       60.290001
AMARK3    12.640000
AMARK1    11.800000
AMARK2    11.680000
Name: Alpha, dtype: float32

Every one of the top hits is a housekeeping gene. This is the failure mode FOCAL exists to
avoid, drawn as bluntly as possible: **magnitude is not identity**. A gene can dominate a
population's expression and still carry no information about what makes that population
different from its neighbours.

## Now FOCAL

One call. `reference="siblings"` means *every cell that is not the target* — here, the other two
populations. Omitting `target` attributes every population in turn, each against its own
reference.

In [4]:
res = focal.attribute(StubEncoder(len(genes)), adata, "state",
                      reference="siblings", device="cpu")

for s in states:
    print(f"{s:6s} ->", res.top(s, 5))

Alpha  -> ['AMARK3', 'AMARK1', 'AMARK2', 'NOISE19', 'NOISE16']
Beta   -> ['BMARK3', 'BMARK1', 'BMARK2', 'NOISE10', 'NOISE9']
Gamma  -> ['GMARK3', 'GMARK1', 'GMARK2', 'NOISE15', 'NOISE20']


All three markers, top-3, for all three populations. The housekeeping genes are gone — not
because they were filtered out, but because they *do not move* along the contrast direction: they
are equally high in the target and in the reference, so they contribute nothing to separating
them.

## The three things you get back

`attribute` returns an `AttributionResult` with three parts worth knowing.

In [5]:
print("1. res.attribution -- raw signed attribution, genes x states")
print("   shape:", res.attribution.shape)
display(res.attribution.head(3).round(4))

print("\n2. res.genes -- every gene, ranked, per state")
print("   lengths:", {s: len(v) for s, v in res.genes.items()}, f"(adata has {adata.n_vars} genes)")

print("\n3. res.top(state, k) -- just the prefix of res.genes")
print("   res.top('Alpha', 3) ==", res.top("Alpha", 3))

1. res.attribution -- raw signed attribution, genes x states
   shape: (39, 3)


,Alpha,Beta,Gamma
HK1,-0.0707,-0.0614,-0.0751
HK2,-0.0581,-0.0872,-0.0604
HK3,-0.0665,-0.0764,-0.0630



2. res.genes -- every gene, ranked, per state
   lengths: {'Alpha': 39, 'Beta': 39, 'Gamma': 39} (adata has 39 genes)

3. res.top(state, k) -- just the prefix of res.genes
   res.top('Alpha', 3) == ['AMARK3', 'AMARK1', 'AMARK2']


:::{important}
`res.genes[state]` contains **all** of `adata.var_names`, not just the genes with positive
attribution. Genes whose attribution is `<= 0` are ranked last rather than dropped, because a
negative attribution is a real statement — it argues for the *reference* direction, not the
target. "The markers" is always a prefix you choose: `res.top(state, k)`.
:::

Because we planted the structure, we can group `Alpha`'s 39 genes by what they *are* and look at
the attribution each class receives:

In [6]:
alpha_rank = {g: i for i, g in enumerate(res.genes["Alpha"])}

def gene_class(g):
    if g in marks["Alpha"]:                        return "Alpha's own markers"
    if any(g in v for v in marks.values()):        return "other populations' markers"
    if g in hk:                                    return "housekeeping (high, shared)"
    return "noise"

summary = pd.DataFrame({"attribution": res.attribution["Alpha"],
                        "rank":  [alpha_rank[g] for g in res.attribution.index],
                        "class": [gene_class(g) for g in res.attribution.index]})
(summary.groupby("class")
        .agg(n=("attribution", "size"), mean_attribution=("attribution", "mean"),
             best_rank=("rank", "min"), worst_rank=("rank", "max"))
        .round(4))

,n,mean_attribution,best_rank,worst_rank
class,,,,
Alpha's own markers,3,0.6501,0,2
"housekeeping (high, shared)",5,-0.0637,13,31
noise,25,-0.0274,3,38
other populations' markers,6,-0.1692,18,23


Read that table row by row — it is the whole method in miniature:

- **Alpha's own markers** take ranks 0–2 with a mean attribution of about `+0.65`. They are the
  only class with a meaningfully positive score.
- **Other populations' markers** carry the *most negative* mean attribution of any class. That is
  not a failure — a `BMARK` gene is positive evidence for Beta, and Beta is the reference here, so
  arguing hard for the reference is precisely what a large negative attribution means.
- **Housekeeping genes** sit near zero despite being the highest-expressed genes in the data. They
  do not move along the contrast, so they do not score. This is the row that separates FOCAL from
  a "top expressed genes" list.

Note that the classes overlap in *rank* (noise spans ranks 3–38) even though they separate cleanly
in mean attribution. Only 4 of 39 genes have positive attribution here, so everything below rank 3
is a ordering of near-zero and negative values — real, but not meaningful. Take a prefix; do not
read the tail.

## Contrast QC comes for free

Since v0.5.0 every `attribute` call also returns two embedding-only diagnostics per state, at
negligible cost. `dprime` is how far apart the target and reference sit along the contrast
direction; `cos_u` is how stable that direction is under random half-splits of the cells.

In [7]:
res.qc.round(3)

,n_target,n_reference,dprime,cos_u_mean,cos_u_min
Alpha,80.0,160.0,9.779,0.979,0.968
Beta,80.0,160.0,10.805,0.980,0.968
Gamma,80.0,160.0,10.371,0.979,0.968


`d' ≈ 10` and `cos_u ≈ 0.98` — a clean, well-separated contrast, which is what you would hope for
in a toy with planted markers. [Tutorial 4](04_contrast_qc) shows what these numbers look like
when the contrast is *not* real, and why that case is dangerous enough to deserve its own
notebook.

## Reweighting the ranking

`focal.composite` re-scores the positive-attribution channel by per-gene expression specificity
and discriminativeness computed from the same `adata`. It needs no encoder and no torch.

In [8]:
n_pos = int((res.attribution["Alpha"] > 0).sum())
print(f"genes with positive attribution for Alpha: {n_pos} of {adata.n_vars}")

k = 3   # only compare within the positive channel -- see the note below
pd.DataFrame({m: focal.composite(res, adata, "state", mode=m)["Alpha"][:k]
              for m in ("bare", "tauE", "tauE_discrRU")},
             index=[f"#{i+1}" for i in range(k)])

genes with positive attribution for Alpha: 4 of 39


,bare,tauE,tauE_discrRU
#1,AMARK3,AMARK3,AMARK3
#2,AMARK1,AMARK1,AMARK1
#3,AMARK2,AMARK2,AMARK2


On a toy this clean the modes agree; on real data they do not, and the differences are the point
— see [Composite modes](../usage.md#composite-modes) for what each one rewards.

:::{warning}
We compared only the top 3 on purpose. Just 4 of these 39 genes have positive attribution, and
`composite` ranks non-positive genes last in *every* mode — so beyond rank 4 you would be
comparing the orderings two functions happen to impose on a set of genes that both consider
uninformative. `res.top()` and `composite(mode="bare")` genuinely disagree there, and neither is
wrong. Choose `k` from your data, not from the length of the list.
:::

## Where to go next

- [Tutorial 2](02_subtype_markers) — the same call against a real foundation model and a real
  25,980-cell atlas, plus the two preprocessing contracts that silently ruin results.
- [Tutorial 3](03_choosing_the_reference) — the argument that makes FOCAL different from a
  differential-expression test.
- [Tutorial 4](04_contrast_qc) — how to tell a real panel from a confident-looking one.